In [ ]:
### LangChain + ChromaDB + Ollama LLM + RAG Application (Step 3)

from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM

e:\Python File\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Initialize Ollama LLM
# Make sure Ollama is running: ollama serve
# Pull models first:
#   ollama pull mistral
#   ollama pull nomic-embed-text

llm = OllamaLLM(
    model="mistral",
    base_url="http://localhost:11434",
    temperature=0.2
)

The charmap (cp1252) codec is the default Windows text encoding, and it cannot represent certain characters — such as:

“Smart quotes” → “ ” ‘ ’

En dashes / em dashes → – —

Non-ASCII punctuation → … • © ®

Emojis or special symbols

In [3]:
# clean_text.py
input_file = "test.txt"
output_file = "clean_test.txt"

with open(input_file, "r", encoding="utf-8", errors="ignore") as f:
    data = f.read()

# Remove unwanted characters (like smart quotes, non-ASCII symbols, etc.)
cleaned_data = ''.join(char for char in data if ord(char) < 128)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(cleaned_data)

print("✅ Clean file saved as:", output_file)


✅ Clean file saved as: clean_test.txt


In [ ]:
loader = TextLoader("clean_test.txt", encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)
print(f"Chunks ready: {len(docs)}")

C:\Users\nitee\AppData\Local\Temp\ipykernel_13172\1986432065.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\nitee\AppData\Local\Temp\ipykernel_13172\1986432065.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


In [ ]:
docsearch = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./data/vector_store/chroma_step3"
)
print("ChromaDB index created locally.")

In [ ]:
query = "What is Logistic Regression?"
found_docs = docsearch.similarity_search(query, k=2)
for i, d in enumerate(found_docs, 1):
    print(f"Result {i}:\n{d.page_content[:400]}\n")

Machine Learning is divided into three main types:
1. Supervised Learning  the model is trained using labeled data. Example: predicting house prices.
2. Unsupervised Learning  the model discovers hidden patterns in unlabeled data. Example: customer segmentation.
3. Reinforcement Learning  the model learns through trial and error by interacting with an environment. Example: training a robot to walk.

Deep Learning (DL) is a specialized branch of machine learning that uses neural networks with many layers (hence deep). Deep learning models are especially powerful for image classification, speech recognition, and natural language processing tasks.

Key Algorithms in Machine Learning:
- Linear Regression
- Logistic Regression
- Decision Trees
- Random Forests
- Support Vector Machines (SVM)
- K-Means Clustering
- Neural Networks


In [ ]:
### Retriever
retriever = docsearch.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001E60001BC90>, search_kwargs={})

In [ ]:
query = "Artificial intelligence in-depth explanation"
for i, d in enumerate(retriever.invoke(query), 1):
    print(f"Retrieved {i}: {d.metadata.get('source', 'unknown')}")
    print(d.page_content[:250], "\n")

Document(metadata={'source': 'clean_test.txt'}, page_content='Introduction to Artificial Intelligence (AI) and Machine Learning\n\nArtificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, problem-solving, perception, and language understanding. AI enables machines to perform cognitive tasks that were once thought to be exclusive to humans.\n\nAI can be categorized into two main types:\n1. Narrow AI  designed for specific tasks like image recognition or language translation.\n2. General AI  aims to perform any intellectual task that a human can do, though it is still largely theoretical.\n\nMachine Learning (ML) is a subset of AI that focuses on the ability of machines to learn from data and improve their performance over time without being explicitly programmed. Machine learning models identify patterns in data and make predictions or decisions based on those patterns.')

## Step 4: Create RAG Chain with Prompt Template

This section builds a full Retrieval-Augmented Generation pipeline:
1. Retrieve relevant chunks from ChromaDB
2. Inject context into a structured prompt
3. Generate final answer with local Ollama LLM

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """You are a helpful support assistant.
Use only the provided context to answer the user question.
If the answer is not in the context, say: 'I could not find that in the provided documents.'

Context:
{context}

Question:
{question}

Answer in clear bullet points when possible."""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

## Execute RAG Queries

In [ ]:
queries = [
    "What is Artificial Intelligence?",
    "Explain machine learning in simple terms.",
    "What is logistic regression used for?"
]

for q in queries:
    print("=" * 80)
    print(f"Question: {q}")
    answer = rag_chain.invoke(q)
    print("\nAnswer:")
    print(answer)
    print()

## Evaluate Retrieved Results

In [ ]:
test_query = "What is machine learning?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Retrieved Document {i} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(doc.page_content[:500])
    print()